In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow xgboost lightgbm prophet

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import pandas as pd
import mlflow

!mlflow db upgrade sqlite:///mlflow.db
mlflow.set_tracking_uri('sqlite:///mlflow.db')

In [ ]:
MODEL_NAME = 'walmart-sales-model'

client = mlflow.MlflowClient()
latest = client.get_registered_model(MODEL_NAME).latest_versions[0]
print(latest.name, 'v' + str(latest.version), latest.run_id)

In [ ]:
model = mlflow.sklearn.load_model(f'models:/{MODEL_NAME}/{latest.version}')
model

In [ ]:
test = pd.read_csv(f'{DATA_DIR}/test.csv', parse_dates=['Date'])
test.shape

In [ ]:
preds = model.predict(test[['Store', 'Dept', 'Date', 'IsHoliday']])
preds[:10]

In [ ]:
submission = pd.DataFrame({
    'Id': test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': preds,
})
submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
if IN_COLAB:
    files.download('submission.csv')

In [ ]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission.csv -m "best model from Model Registry"